# nanoGPT — Build a Transformer from Scratch

**What you'll learn:** how self-attention works at the equation level, how a transformer block is assembled, and why GPT generates text one token at a time.

**Estimated time:** 90–120 min &nbsp;|&nbsp; **Runtime:** GPU

---

### Read this before touching any code

Everything in this notebook flows from one idea: **attention**. The model needs to predict the next character. To do that well, it needs to look at previous characters and decide which ones are relevant.

Self-attention formalizes this as:
```
Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
```

- **Q (Query):** "What am I looking for?"
- **K (Key):** "What do I contain?"
- **V (Value):** "What should I pass forward if attended to?"

Every position computes its own Q, K, V. Positions with similar Q and K vectors (high dot product) attend to each other strongly.

? **Before any code:** In a sentence like "The cat sat on the mat", which word do you think "mat" would attend most strongly to — "cat", "sat", or "on"? Why?


## 1  Data: turn text into numbers

In [ ]:
import torch, torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# A small corpus. Replace with any longer .txt for better results.
text = (
    'to be or not to be that is the question whether tis nobler in the mind '
    'to suffer the slings and arrows of outrageous fortune or to take arms against '
    'a sea of troubles and by opposing end them to die to sleep no more and by a '
    'sleep to say we end the heartache and the thousand natural shocks that flesh '
    'is heir to tis a consummation devoutly to be wished to die to sleep to sleep '
    'perchance to dream ay there is the rub for in that sleep of death what dreams '
    'may come when we have shuffled off this mortal coil must give us pause '
) * 6

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s if c in stoi]
decode = lambda l: ''.join(itos.get(i, '') for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Corpus: {len(text)} chars -> {len(data)} tokens")
print(f"Vocabulary: {vocab_size} unique characters: {chars[:20]}...")

### How does tokenization work?

Each character gets a unique integer ID. The model never sees text — only sequences of integers.

? **Run the next cell and read the output carefully.** What does the model's training task look like at this level?


In [ ]:
# Show what a training example looks like
sample_text = text[:20]
sample_ids  = encode(sample_text)
print("Text:  ", repr(sample_text))
print("IDs:   ", sample_ids)
print()
print("Training signal: for each position, predict the next token")
for i in range(min(8, len(sample_ids) - 1)):
    context = decode(sample_ids[:i+1])
    target  = decode([sample_ids[i+1]])
    print(f"  context={repr(context):20s}  ->  next={repr(target)}")

## 2  Hyperparameters and batching

In [ ]:
# Small model that trains in ~2 min on GPU
block_size = 32    # how many characters of context the model sees at once
batch_size = 64    # how many independent sequences to train on simultaneously
n_embd     = 64    # embedding dimension (size of each token's vector)
n_head     = 4     # number of attention heads
n_layer    = 3     # number of transformer blocks stacked

# 90/10 train/val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

def get_batch(split='train'):
    d = train_data if split == 'train' else val_data
    # Sample random starting positions
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i + block_size]     for i in ix])  # input
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # target (shifted by 1)
    return x.to(device), y.to(device)

xb, yb = get_batch()
print("Input  shape:", xb.shape)   # (batch_size, block_size)
print("Target shape:", yb.shape)   # same — each position predicts the next

? **Think:** `y` is `x` shifted by 1 position. Why? What is the model being asked to predict?


## 3  Self-attention — the core mechanism

This is the most important part of the notebook. Read every line.

Each head projects the input into Q, K, V spaces. Then:
1. Compute attention scores: `Q @ Kᵀ / √head_size`
2. Mask future positions (a character can't attend to tokens it hasn't seen yet — that would be cheating)
3. Softmax to get attention weights
4. Weighted sum of V

Fill in the forward pass below.


In [ ]:
class Head(nn.Module):
    # Single self-attention head
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Causal mask: lower-triangular matrix of 1s
        # tril[i][j] = 1 means position i is allowed to attend to position j
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape       # batch, time (sequence length), channels (n_embd)
        head_size = self.key.out_features

        # Step 1: project to Q, K, V
        # YOUR CODE HERE
        k = self.key(x)         # (B, T, head_size)
        q = self.query(x)       # (B, T, head_size)
        v = self.value(x)       # (B, T, head_size)

        # Step 2: compute raw attention scores
        # Q @ K^T gives (B, T, T) — how much each position attends to each other
        # Scale by 1/sqrt(head_size) to stabilize gradients (without this, softmax saturates)
        # YOUR CODE HERE
        att = q @ k.transpose(-2, -1) * head_size**-0.5    # (B, T, T)

        # Step 3: apply the causal mask — zero out future positions
        # Replace masked positions with -inf so softmax gives them 0 weight
        att = att.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # Step 4: softmax over the key dimension
        att = F.softmax(att, dim=-1)  # (B, T, T)

        # Step 5: weighted sum of values
        # YOUR CODE HERE
        return att @ v           # (B, T, head_size)

### Why the causal mask?

Without it, position 5 could attend to position 6 (a future character). During generation, those future tokens don't exist yet — the mask enforces the autoregressive constraint.

? **Visualize the mask:** Run the next cell and look at it. What pattern do you see?


In [ ]:
# Visualize the causal mask
import matplotlib.pyplot as plt
mask = torch.tril(torch.ones(8, 8))
plt.figure(figsize=(4, 4))
plt.imshow(mask, cmap='Blues', vmin=0, vmax=1)
plt.title('Causal mask (8x8 example)')
plt.xlabel('Key position (past -> future)')
plt.ylabel('Query position')
for i in range(8):
    for j in range(8):
        plt.text(j, i, '✓' if mask[i, j] else '✗', ha='center', va='center', fontsize=10)
plt.tight_layout()
plt.show()
print("Row i = position i can see columns up to and including i, but not beyond.")

## 4  Multi-head attention and the transformer block

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj  = nn.Linear(n_embd, n_embd)   # mix the heads' outputs

    def forward(self, x):
        # Run all heads in parallel, concatenate along the last dim
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)


class FeedForward(nn.Module):
    # Position-wise MLP: expands 4x then contracts back
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )
    def forward(self, x): return self.net(x)


class Block(nn.Module):
    # Transformer block: attention -> residual -> FFN -> residual
    def __init__(self):
        super().__init__()
        head_size = n_embd // n_head
        self.sa  = MultiHeadAttention(n_head, head_size)
        self.ff  = FeedForward()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Pre-norm residual connections: normalize -> transform -> add back
        # YOUR CODE HERE — write both residual connections
        x = x + self.sa(self.ln1(x))    # attention with residual
        x = x + self.ff(self.ln2(x))    # FFN with residual
        return x

? **What is the residual connection doing?** (`x = x + transform(x)`)

The residual connection lets the gradient flow directly from the output all the way back to the early layers — without passing through every transformation in between. It's why deep networks are trainable.


## 5  The full GPT model

In [ ]:
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)   # char -> vector
        self.pos_emb   = nn.Embedding(block_size, n_embd)   # position -> vector
        self.blocks    = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f      = nn.LayerNorm(n_embd)
        self.lm_head   = nn.Linear(n_embd, vocab_size)      # -> logit per char

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # Token embedding + positional embedding
        tok = self.token_emb(idx)                                   # (B, T, n_embd)
        pos = self.pos_emb(torch.arange(T, device=device))         # (T, n_embd)
        x = tok + pos                                               # (B, T, n_embd)

        x = self.lm_head(self.ln_f(self.blocks(x)))                # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Reshape for cross-entropy: (B*T, vocab_size) vs (B*T,)
            loss = F.cross_entropy(x.view(-1, vocab_size), targets.view(-1))
        return x, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature    # last position
            probs  = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

model = GPT().to(device)
total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total:,}")
print(f"(~{total/1e6:.1f}M — tiny by modern standards, but all the same ideas)")

## 6  Before training — baseline generation

? **What do you expect the model to output before any training?** Run the next cell and observe.


In [ ]:
# Untrained generation
context = torch.zeros((1, 1), dtype=torch.long, device=device)
output = model.generate(context, max_new_tokens=100)[0].tolist()
print("Before training:")
print(repr(decode(output)))
print()
print("(Should look like random characters — the model knows nothing yet)")

## 7  Train

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        ls = [model(*get_batch(split))[1].item() for _ in range(20)]
        losses[split] = sum(ls) / len(ls)
    model.train()
    return losses

print("Training...")
for step in range(3000):
    xb, yb = get_batch()
    _, loss = model(xb, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        ev = estimate_loss()
        print(f"Step {step:4d}: train={ev['train']:.3f}  val={ev['val']:.3f}")

## 8  After training — generate text

In [ ]:
# Greedy generation (temperature=1.0 is default sampling)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
for temp in [0.7, 1.0, 1.5]:
    output = model.generate(context, max_new_tokens=150, temperature=temp)[0].tolist()
    print(f"Temperature={temp}:")
    print(decode(output))
    print()

? **Reflect on the temperature results:**
- At temperature 0.7 (lower): the output is more... repetitive or creative?
- At temperature 1.5 (higher): what happens?

Temperature scales the logits before softmax. Low temperature makes the probability distribution sharper (model picks its top choice more often). High temperature flattens it (more random).

This is the same `temperature` parameter you see in every LLM API.


## 9  Challenge: understand the model you built

Answer these in code or writing:

1. **Count attention operations:** How many Q@K^T matrix multiplications happen in a single forward pass? (Think: blocks x heads x ...)

2. **Upload Shakespeare:** Download [tinyshakespeare](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt) and retrain. How does the quality change?

3. **Annotate the Head:** Go through `Head.forward` line by line and add a comment explaining what each line does in plain English.

4. **Ablation — remove positional embeddings:** Set `pos = 0` in the forward pass. What happens to training? Why does position information matter?

```python
# Starter for ablation
pos = 0  # no positional info
x = tok + pos
```

The insights you build here — residuals, causal masking, temperature — are exactly how GPT-4 and Claude work at scale.
